In [ ]:
# Run the cells from top to bottom in Google Colab. No API key is needed.
%pip install -q langgraph

In [ ]:
from operator import add
from typing import Annotated, Literal, NotRequired, TypedDict

from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: NotRequired[Literal["billing", "technical"]]
    reply: NotRequired[str]
    # Each node returns only its new entry. The reducer appends it to the list.
    history: Annotated[list[str], add]


def classify(state: TicketState) -> dict:
    billing_keywords = ("invoice", "payment", "refund", "charged", "billing")
    ticket = state["ticket"].casefold()
    category = "billing" if any(word in ticket for word in billing_keywords) else "technical"
    return {"category": category, "history": ["classify"]}


def route_ticket(state: TicketState) -> Literal["billing", "technical"]:
    return state["category"]


def billing(state: TicketState) -> dict:
    return {
        "reply": "Please share your invoice number so we can look up the charge.",
        "history": ["billing"],
    }


def technical(state: TicketState) -> dict:
    return {
        "reply": "Please share the error message and the steps that led to it.",
        "history": ["technical"],
    }


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_ticket,
    {"billing": "billing", "technical": "technical"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)
graph = builder.compile()

In [ ]:
# Expected history: ["classify", "billing"]
billing_result = graph.invoke({"ticket": "Can I get a refund?", "history": []})
print("Category:", billing_result["category"])
print("Reply:", billing_result["reply"])
print("History:", billing_result["history"])

In [ ]:
# Expected history: ["classify", "technical"]
technical_result = graph.invoke({"ticket": "The app crashes when I open it.", "history": []})
print("Category:", technical_result["category"])
print("Reply:", technical_result["reply"])
print("History:", technical_result["history"])

In [ ]:
from IPython.display import Image, display

diagram = graph.get_graph()

# The online Mermaid renderer receives the graph structure only.
try:
    display(Image(diagram.draw_mermaid_png()))
except Exception:
    # Show the diagram source if the image service is unavailable.
    print(diagram.draw_mermaid())